In [3]:
!git clone -b feature/backtest_shryes https://github.com/rao-shreyashree/demand-forecasting.git
%cd /content/demand-forecasting
!git status

Cloning into 'demand-forecasting'...
remote: Enumerating objects: 232, done.
remote: Counting objects: 100% (232/232), done.
remote: Compressing objects: 100% (163/163), done.
remote: Total 232 (delta 109), reused 151 (delta 56), pack-reused 0 (from 0)
Receiving objects: 100% (232/232), 3.59 MiB | 12.54 MiB/s, done.
Resolving deltas: 100% (109/109), done.
/content/demand-forecasting
On branch feature/backtest_shryes
Your branch is up to date with 'origin/feature/backtest_shryes'.

nothing to commit, working tree clean


In [5]:
!git fetch origin
!git config user.name Shryes19
!git config user.email shryes107@gmail.com
!git -c core.editor=true merge --no-edit origin/main

Already up to date.


In [6]:
%cd /content/demand-forecasting

!git status
!git log -1 --oneline
!ls data/processed

/content/demand-forecasting
On branch feature/backtest_shryes
Your branch is ahead of 'origin/feature/backtest_shryes' by 14 commits.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean
e389c76 (HEAD -> feature/backtest_shryes) Merge remote-tracking branch 'origin/main' into feature/backtest_shryes
forecasts_classical_backtest.csv  forecasts_ml.csv
forecasts_classical.csv		  inventory_plan.csv


In [7]:
import os
import sys
import pandas as pd
import numpy as np

REPO_ROOT = os.getcwd()

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.schema import DATE_COL, STORE_COL, ITEM_COL, SALES_COL
from src.baselines import forecast_by_group
from src.backtest import rolling_origin_windows, evaluate_forecast, run_backtest

print("Repository:", REPO_ROOT)
print("All imports OK")

Repository: /content/demand-forecasting
All imports OK


In [8]:
df_raw = pd.read_csv("data/raw/train.csv")

df_raw[DATE_COL] = pd.to_datetime(df_raw[DATE_COL])

df = (
    df_raw
    .sort_values([STORE_COL, ITEM_COL, DATE_COL])
    .reset_index(drop=True)
)

print("Loaded:", df.shape)
print("Date range:", df[DATE_COL].min(), "→", df[DATE_COL].max())
print("Unique store-item pairs:", df[[STORE_COL, ITEM_COL]].drop_duplicates().shape[0])

Loaded: (913000, 4)
Date range: 2013-01-01 00:00:00 → 2017-12-31 00:00:00
Unique store-item pairs: 500


In [9]:
windows = rolling_origin_windows(df, horizon=28, n_windows=3)

for w in windows:
    print(
        f"Window {w['window']}: "
        f"validation {w['val_start'].date()} → {w['val_end'].date()} | "
        f"train rows: {len(w['train'])} | "
        f"validation rows: {len(w['val'])}"
    )

Window 3: validation 2017-10-09 → 2017-11-05 | train rows: 871000 | validation rows: 14000
Window 2: validation 2017-11-06 → 2017-12-03 | train rows: 885000 | validation rows: 14000
Window 1: validation 2017-12-04 → 2017-12-31 | train rows: 899000 | validation rows: 14000


In [10]:
# Load the classical model forecasts produced by the merged main branch
classical_bt = pd.read_csv("data/processed/forecasts_classical_backtest.csv")
classical_bt[DATE_COL] = pd.to_datetime(classical_bt[DATE_COL])

# Identify the exact 10 store-item pairs used by Prophet and SARIMA
top10_keys = classical_bt[[STORE_COL, ITEM_COL]].drop_duplicates()

assert len(top10_keys) == 10, (
    f"Expected 10 store-item pairs, found {len(top10_keys)}"
)

print("Classical forecasts loaded:", classical_bt.shape)
print("Common store-item pairs:", len(top10_keys))


def combined_forecast(train_df, val_df):
    # The validation dates for this particular rolling window
    val_dates = pd.DatetimeIndex(sorted(val_df[DATE_COL].unique()))
    horizon = len(val_dates)

    # Keep only classical forecasts belonging to this validation window
    classical_window = classical_bt[
        classical_bt[DATE_COL].isin(val_dates)
    ].copy()

    assert len(classical_window) == 560, (
        f"Expected 560 classical forecast rows, got {len(classical_window)}"
    )

    # Confirm classical forecasts use exactly the common 10 pairs
    classical_keys = classical_window[
        [STORE_COL, ITEM_COL]
    ].drop_duplicates()

    assert set(map(tuple, classical_keys.to_numpy())) == set(
        map(tuple, top10_keys.to_numpy())
    ), "Classical forecasts do not contain the expected 10 pairs"

    # Restrict this window's training data to those same 10 pairs
    train_top10 = train_df.merge(
        top10_keys,
        on=[STORE_COL, ITEM_COL],
        how="inner"
    )

    # Generate all baselines using ONLY this window's training data
    baseline_specs = [
        ("naive", {}),
        ("seasonal_naive", {"season": 7}),
        ("moving_average", {"window": 7}),
        ("moving_average", {"window": 28}),
    ]

    baseline_forecasts = []

    for method, kwargs in baseline_specs:
        forecast = forecast_by_group(
            train_top10,
            method=method,
            horizon=horizon,
            **kwargs
        )

        # Check that each baseline forecasts all 10 pairs for all 28 days
        assert len(forecast) == 280, (
            f"{method} {kwargs}: expected 280 rows, got {len(forecast)}"
        )

        assert set(pd.to_datetime(forecast[DATE_COL]).unique()) == set(val_dates), (
            f"{method} {kwargs}: forecast dates do not match validation dates"
        )

        baseline_forecasts.append(forecast)

    # Combine Prophet, SARIMA and the four freshly generated baselines
    combined = pd.concat(
        [classical_window] + baseline_forecasts,
        ignore_index=True
    )

    combined[DATE_COL] = pd.to_datetime(combined[DATE_COL])

    return combined[
        [DATE_COL, STORE_COL, ITEM_COL, "y_hat", "model"]
    ]


# Run the fair comparison across all three rolling windows
comparison_table, detailed = run_backtest(
    combined_forecast,
    df,
    n_windows=3,
    horizon=28
)

print("\nComparison across the three windows:")
display(comparison_table)

Classical forecasts loaded: (1680, 5)
Common store-item pairs: 10

Comparison across the three windows:


,model,wape,rmse,mape,bias,n_rows,n_windows,rank
0,prophet,0.0892,12.2081,0.0938,1.6336,840,3,1
1,sarima,0.1153,15.7683,0.1214,4.3679,840,3,2
2,seasonal_naive_7,0.1567,20.8836,0.1680,5.0738,840,3,3
3,ma_7,0.1584,21.0238,0.1717,5.0738,840,3,4
4,ma_28,0.2057,25.0588,0.2234,11.1476,840,3,5
5,naive,0.2349,31.3656,0.2618,23.0262,840,3,6


In [11]:
# Save the final comparison outputs
import os

os.makedirs("data/processed", exist_ok=True)

comparison_table.to_csv(
    "data/processed/model_comparison_shryes.csv",
    index=False
)

detailed.to_csv(
    "data/processed/backtest_detailed_shryes.csv",
    index=False
)

print("Saved comparison and detailed backtest results.")
print("Comparison shape:", comparison_table.shape)
print("Detailed results shape:", detailed.shape)

Saved comparison and detailed backtest results.
Comparison shape: (6, 8)
Detailed results shape: (18, 7)


In [12]:
!pytest -q

........................................................................ [ 53%]
..............................................................           [100%]
134 passed in 9.08s


In [13]:
!git status
!git diff --stat
!git branch --show-current

On branch feature/backtest_shryes
Your branch is ahead of 'origin/feature/backtest_shryes' by 14 commits.
  (use "git push" to publish your local commits)

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	scripts/__pycache__/
	src/__pycache__/
	tests/__pycache__/

nothing added to commit but untracked files present (use "git add" to track)
feature/backtest_shryes


In [14]:
!git status --short
!git check-ignore -v notebooks/backtest_shryes.ipynb
!git diff -- notebooks/backtest_shryes.ipynb

?? scripts/__pycache__/
?? src/__pycache__/
?? tests/__pycache__/


In [15]:
import os

print("Notebook exists:", os.path.exists("notebooks/backtest_shryes.ipynb"))
print("Notebook size:", os.path.getsize("notebooks/backtest_shryes.ipynb"))

Notebook exists: True
Notebook size: 13815


In [16]:
import json

path = "notebooks/backtest_shryes.ipynb"

with open(path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

print("Total cells:", len(notebook["cells"]))

for i, cell in enumerate(notebook["cells"]):
    if cell["cell_type"] == "code":
        source = "".join(cell["source"])
        print(f"\n--- Cell {i} ---")
        print(source[:500])

print("\nContains obsolete baseline file reference:",
      any("forecasts_baselines.csv" in "".join(c.get("source", []))
          for c in notebook["cells"]))

print("\nContains dynamic baseline function:",
      any("def combined_forecast" in "".join(c.get("source", []))
          for c in notebook["cells"]))

Total cells: 11

--- Cell 0 ---
import pandas as pd
import numpy as np
import sys
sys.path.append('..')

from src.schema import DATE_COL, STORE_COL, ITEM_COL, SALES_COL
from src.data import load_data, clean_data, time_split
from src.metrics import wape, rmse, mape, bias

from src.backtest import rolling_origin_windows, evaluate_forecast, run_backtest

print("All imports OK")

--- Cell 1 ---
df_raw = pd.read_csv('../data/raw/train.csv')
df_raw[DATE_COL] = pd.to_datetime(df_raw[DATE_COL])
df = df_raw.sort_values([STORE_COL, ITEM_COL, DATE_COL]).reset_index(drop=True)
print("Loaded:", df.shape)
print("Date range:", df[DATE_COL].min(), "→", df[DATE_COL].max())

--- Cell 2 ---
def rolling_origin_windows(df, horizon=28, n_windows=3):
    """
    Generates n_windows (train, val) splits.
    Each val window is 'horizon' days.
    Windows roll forward — val of window i becomes part of train in window i+1.
    """
    all_dates = sorted(df[DATE_COL].unique())
    total_days = len(all_dates)
    